# Notebook 2: a decorrelated second encoder

Two experiments have now said the same thing. `v12` gained by **adding** a TabPFN arm
trained on differently-shaped input and lost when it **replaced** one. Notebook 1 built an
arm that was the same model on nearly the same features and it contributed nothing, with
the optimiser handing it weight 0.0 on CYP2C9. This blend is not short of feature quality.
It is short of disagreement.

So: a second pretrained encoder. `chemprop_chemeleon` is CheMeleon-foundation-initialised
and then trained on the same corpus as `chemprop_medium`, which every arm we ship already
runs on. The entry at live rank 48 calls it "a genuinely decorrelated second cluster, not
a duplicate", and their rule is **freeze, don't fine-tune**, with a stated mechanism: their
ChEMBL CYP data runs 1 to 1.65 log units more potent than this challenge's screening
population, so fine-tuning pulls the encoder toward a population it will not be scored on.
That explains our own CheMeleon, MoLFormer and ChemBERTa failures, every one of which was
a fine-tune or a frozen-embedding-plus-separate-model, never a frozen second arm.

**Built-in control.** This notebook also re-embeds with `chemprop_medium` and checks the
result against the cached embedding we have been using since October. If those two do not
agree, the extraction is wrong and nothing else here means anything.

**Run order matters.** Chemprop and TabPFN disagree about dependencies. Cell 4 installs
chemprop and writes the embeddings to disk; cell 7 installs TabPFN afterwards. If Colab
demands a restart between them, the embeddings are already saved, so re-run the payload
cell and carry on from cell 7.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU')

In [ ]:
# Challenge data, folds, holdout mask and the cached chemprop_medium embedding.
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
os.makedirs('/content/results', exist_ok=True)
OUTDIR = '/content/results'
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
print(sorted(os.listdir('/content/cache')))

## Embed, with chemprop only

In [ ]:
!pip -q install chemprop==2.3.1

In [ ]:
import numpy as np, pandas as pd, torch, time
from pathlib import Path

CACHE = Path('/content/cache'); OUT = Path(OUTDIR)
meta = pd.read_csv(CACHE / 'meta_train.csv.gz'); mete = pd.read_csv(CACHE / 'meta_test.csv.gz')
SM_TR = meta['SMILES'].tolist(); SM_TE = mete['SMILES'].tolist()

BASE = ('https://raw.githubusercontent.com/jeremycheminf/openadmet_scripts/'
        'main/CYP_Challenge/checkpoints')
import urllib.request
for f in ['chemprop_medium.pt', 'chemprop_chemeleon.pt']:
    p = Path(f'/content/{f}')
    if not p.exists():
        urllib.request.urlretrieve(f'{BASE}/{f}', p)
    print(f, round(p.stat().st_size / 1e6, 1), 'MB')

from chemprop import data, featurizers
from chemprop.models.model import MPNN

def embed(ckpt, smiles, batch_size=256):
    model = MPNN.load_from_file(ckpt); model.eval()
    fz = featurizers.SimpleMoleculeMolGraphFeaturizer()
    pts = [data.MoleculeDatapoint.from_smi(s, [0.0]) for s in smiles]
    dset = data.MoleculeDataset(pts, fz)
    # drop_last=False is required: chemprop's default silently drops a size-1
    # remainder batch, which desynchronises every index-aligned array downstream.
    loader = data.build_dataloader(dset, num_workers=0, batch_size=batch_size,
                                   shuffle=False, drop_last=False)
    out = []
    with torch.inference_mode():
        for batch in loader:
            bmg, V_d, X_d, *_ = batch
            out.append(model.fingerprint(bmg, V_d, X_d).numpy())
    r = np.concatenate(out, 0)
    assert len(r) == len(smiles), f'{len(r)} embedded, {len(smiles)} expected'
    return r.astype(np.float32)

for name, ck in [('medium', 'chemprop_medium.pt'), ('cheme', 'chemprop_chemeleon.pt')]:
    t0 = time.time()
    a, b = embed(f'/content/{ck}', SM_TR), embed(f'/content/{ck}', SM_TE)
    np.save(OUT / f'emb_{name}_train.npy', a); np.save(OUT / f'emb_{name}_test.npy', b)
    print(f'{name}: train {a.shape} test {b.shape} ({time.time()-t0:.0f}s)', flush=True)

In [ ]:
# Control: does re-embedding reproduce the cached chemprop_medium we have shipped since
# October? Compare column by column; PCA and TabPFN are invariant to column order but not
# to the embedding being a different thing entirely.
ref = np.load(CACHE / 'emb_medium_inhib.npy').astype(np.float32)
new = np.load(OUT / 'emb_medium_train.npy')
print('cached', ref.shape, '| re-embedded', new.shape)
if ref.shape == new.shape:
    cor = np.array([np.corrcoef(ref[:, j], new[:, j])[0, 1]
                    for j in range(ref.shape[1]) if ref[:, j].std() > 0])
    print(f'per-column correlation: min {np.nanmin(cor):.4f}, median {np.nanmedian(cor):.4f}')
    print('CONTROL OK' if np.nanmedian(cor) > 0.99 else 'CONTROL FAILED, stop here')
else:
    print('shape mismatch, the extraction is not reproducing the cached embedding')

che = np.load(OUT / 'emb_cheme_train.npy')
print('\nhow much do the two encoders actually disagree?')
k = min(ref.shape[1], che.shape[1])
z = lambda M: (M - M.mean(0)) / (M.std(0) + 1e-9)
cross = np.abs(z(ref[:, :k]).T @ z(che[:, :k])) / len(ref)
print(f'mean |correlation| between their columns: {cross.mean():.4f}')
print('(a duplicate encoder would sit near 1; a decorrelated one well below)')

## TabPFN arms

Raw features, no PCA, the configuration that gained +0.0125 and is now confirmed on the
live board. `medium` is the control arm and should land near the 0.5881 / 0.7060 / 0.4448 /
0.8115 that notebook 1 produced for the same feature set.

In [ ]:
!pip -q install tabpfn==2.2.1

In [ ]:
import json, time, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from tabpfn import TabPFNRegressor

CACHE = Path('/content/cache'); OUT = Path('/content/results')
ISOFORMS = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
N_FOLDS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
KW = dict(device=DEVICE, random_state=42, ignore_pretraining_limits=True)

meta = pd.read_csv(CACHE / 'meta_train.csv.gz')
hold = np.load(CACHE / 'holdout_mask.npy')
G = np.load(CACHE / 'scaffold_groups.npy', allow_pickle=True)
QH, QH_TE = np.load(CACHE / 'qhts_tr.npy'), np.load(CACHE / 'qhts_te.npy')
AUX, AUX_TE = np.load(CACHE / 'aux_train.npy'), np.load(CACHE / 'aux_test.npy')
MED, MED_TE = np.load(OUT / 'emb_medium_train.npy'), np.load(OUT / 'emb_medium_test.npy')
CHE, CHE_TE = np.load(OUT / 'emb_cheme_train.npy'), np.load(OUT / 'emb_cheme_test.npy')
SHIPPED = {'CYP1A2': 0.5791, 'CYP2C9': 0.7076, 'CYP2D6': 0.4263, 'CYP3A4': 0.8094}

def dev_index(iso):
    y = meta[f'{iso}_pIC50_direct_inhibition'].to_numpy(float)
    obs = np.isfinite(y)
    return y, np.flatnonzero(obs & ~hold), np.flatnonzero(obs & hold)

RESULTS = []
ARMS = {'medium': (np.hstack([MED, QH, AUX]), np.hstack([MED_TE, QH_TE, AUX_TE])),
        'cheme':  (np.hstack([CHE, QH, AUX]), np.hstack([CHE_TE, QH_TE, AUX_TE])),
        'both':   (np.hstack([MED, CHE, QH, AUX]),
                   np.hstack([MED_TE, CHE_TE, QH_TE, AUX_TE]))}

for name, (Xtr, Xte) in ARMS.items():
    Xtr = np.nan_to_num(Xtr.astype(np.float32)); Xte = np.nan_to_num(Xte.astype(np.float32))
    print(f'\n{name}  {Xtr.shape[1]} columns', flush=True)
    for iso in ISOFORMS:
        y, dev, ho = dev_index(iso)
        t0 = time.time()
        oof = np.full(len(dev), np.nan)
        for tr, va in GroupKFold(N_FOLDS).split(dev, dev, G[dev]):
            m = TabPFNRegressor(**KW).fit(Xtr[dev[tr]], y[dev[tr]])
            oof[va] = m.predict(Xtr[dev[va]])
        m = TabPFNRegressor(**KW).fit(Xtr[dev], y[dev]); hp = m.predict(Xtr[ho])
        allr = np.concatenate([dev, ho])
        m2 = TabPFNRegressor(**KW).fit(Xtr[allr], y[allr]); tp = m2.predict(Xte)
        np.savez_compressed(OUT / f'enc_{name}_{iso}.npz',
                            oof=oof, dev=dev, hold=hp, ho=ho, test=tp)
        d = float(spearmanr(y[dev], oof).statistic)
        h = float(spearmanr(y[ho], hp).statistic)
        RESULTS.append(dict(arm=name, iso=iso, dev_spearman=d, holdout_spearman=h,
                            shipped=SHIPPED[iso], seconds=round(time.time() - t0, 1)))
        json.dump(RESULTS, open(OUT / 'results.json', 'w'), indent=2)
        print(f'  {iso}: dev {d:.4f} | holdout {h:.4f}  ({time.time()-t0:.0f}s)', flush=True)

In [ ]:
r = pd.DataFrame(RESULTS)
for col, label in [('dev_spearman', 'DEV out-of-fold'), ('holdout_spearman', 'HOLDOUT')]:
    t = r.pivot(index='arm', columns='iso', values=col); t['MACRO'] = t.mean(axis=1)
    print(label); print(t.round(4).to_string()); print()
print('notebook 1 reference for the medium arm: 0.5881 0.7060 0.4448 0.8115 (dev)')

import shutil
from google.colab import files
shutil.make_archive('/content/notebook2_results', 'zip', OUT)
files.download('/content/notebook2_results.zip')